# Measures and Aggregations

In SLayer, a **measure** is a named row-level SQL expression. The **aggregation** (sum, avg, count, etc.) is specified at query time using **colon syntax**: `revenue:sum`, `*:count`, `price:weighted_avg(weight=quantity)`.

This notebook demonstrates the full range of aggregation features with working code.

**Prerequisites:** `pip install motley-slayer[examples]` (jafgen is installed by the cell below)

In [ ]:
# Install jafgen (Jaffle Shop data generator) from a specific commit
# The released PyPI version has a bug; this pinned commit is the fix.
# This is only needed for running the tutorials — not a SLayer dependency.
!pip install -q git+https://github.com/rossbowen/jaffle-shop-generator.git@09557a1118b000071f8171aa97d54d5029bf0f0b

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

engine, storage, models = ensure_jaffle_shop()

## Basic Colon Syntax

Every field formula specifies both the measure and the aggregation, separated by a colon. `*:count` is the universal row count.

In [ ]:
result = engine.execute(
    query={
        "source_model": "orders",
        "fields": ["*:count", "order_total:sum", "order_total:avg", "order_total:min", "order_total:max"],
    }
)

row = result.data[0]
print(f"Orders:  {row['orders._count']:,}")
print(f"Total:   ${row['orders.order_total_sum']:,.2f}")
print(f"Average: ${row['orders.order_total_avg']:.2f}")
print(f"Min:     ${row['orders.order_total_min']:.2f}")
print(f"Max:     ${row['orders.order_total_max']:.2f}")

## Multiple Aggregations on One Measure

One measure, many aggregations — no need to define `order_total_sum`, `order_total_avg`, etc. as separate measures.

In [ ]:
result = engine.execute(
    query={
        "source_model": "orders",
        "fields": ["*:count", "order_total:sum", "order_total:avg"],
        "dimensions": ["stores.name"],
        "order": [{"column": "order_total_sum", "direction": "desc"}],
        "limit": 5,
    }
)

print(f"{'Store':<20} {'Orders':>8} {'Total':>14} {'Average':>10}")
print("-" * 54)
for row in result.data:
    print(
        f"{row['orders.stores.name']:<20} {row['orders._count']:>8,} ${row['orders.order_total_sum']:>13,.2f} ${row['orders.order_total_avg']:>9.2f}"
    )

## Arithmetic on Aggregated Measures

Aggregated measures compose naturally in arithmetic expressions.

In [ ]:
result = engine.execute(
    query={
        "source_model": "orders",
        "fields": [
            "*:count",
            "order_total:sum",
            {"formula": "order_total:sum / *:count", "name": "aov", "label": "Avg Order Value"},
        ],
        "dimensions": ["stores.name"],
        "order": [{"column": "order_total_sum", "direction": "desc"}],
        "limit": 5,
    }
)

for row in result.data:
    print(f"{row['orders.stores.name']}: AOV = ${row['orders.aov']:.2f}")

## Transforms on Aggregated Measures

All transform functions (`cumsum`, `change`, `time_shift`, `rank`, etc.) wrap aggregated measure refs.

In [ ]:
result = engine.execute(
    query={
        "source_model": "orders",
        "fields": [
            "order_total:sum",
            {"formula": "cumsum(order_total:sum)", "name": "running_total"},
            {"formula": "change(order_total:sum)", "name": "mom_change"},
        ],
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

print(f"{'Month':<12} {'Revenue':>14} {'Running Total':>14} {'MoM Change':>12}")
print("-" * 54)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    rev = row["orders.order_total_sum"]
    running = row["orders.running_total"]
    change = row["orders.mom_change"]
    change_str = f"${change:>11,.2f}" if change is not None else f"{'N/A':>12}"
    print(f"{month:<12} ${rev:>13,.2f} ${running:>13,.2f} {change_str}")

## Cross-Model Aggregated Measures

Cross-model measures use dot syntax for the join path, colon for the aggregation. `customers.*:count` means COUNT(*) on the joined customers model.

In [ ]:
result = engine.execute(
    query={
        "source_model": "orders",
        "fields": [
            "*:count",
            "order_total:sum",
            "customers.*:count",
        ],
        "time_dimensions": [{"dimension": "ordered_at", "granularity": "month"}],
        "order": [{"column": "ordered_at", "direction": "asc"}],
        "limit": 6,
    }
)

print(f"{'Month':<12} {'Orders':>8} {'Revenue':>14} {'Customers':>10}")
print("-" * 46)
for row in result.data:
    month = str(row["orders.ordered_at"])[:7]
    orders = row["orders._count"]
    rev = row["orders.order_total_sum"]
    custs = row["orders.customers._count"]
    print(f"{month:<12} {orders:>8,} ${rev:>13,.2f} {custs:>10,}")

## Inspecting Model Measures

Auto-ingested models now have one measure per column — not five.

In [ ]:
orders_model = next(m for m in models if m.name == "orders")

print(f"orders model has {len(orders_model.measures)} measures:")
for m in orders_model.measures:
    print(f"  {m.name:<20} sql: {m.sql}")

print("\nAll of these can be aggregated any way you want at query time.")
print("Plus *:count is always available for COUNT(*).")

## Custom Aggregation: Weighted Average

Let's define a model with a custom weighted average aggregation.

In [ ]:
from slayer.core.models import Aggregation, AggregationParam, Measure, SlayerModel, Dimension

# Create a model with a custom weighted_avg that defaults weight to quantity
items_model = SlayerModel(
    name="order_items_custom",
    sql_table="order_items",
    data_source="jaffle_shop",
    dimensions=[
        Dimension(name="sku", sql="sku", type="string"),
    ],
    measures=[
        Measure(name="quantity", sql="quantity"),
    ],
    aggregations=[
        Aggregation(
            name="weighted_avg",
            params=[AggregationParam(name="weight", sql="quantity")],
        ),
    ],
)

# Save and query
storage.save_model(items_model)

result = engine.execute(
    query={
        "source_model": "order_items_custom",
        "fields": [
            "*:count",
            "quantity:sum",
            "quantity:avg",
        ],
    }
)

row = result.data[0]
print(f"Line items: {row['order_items_custom._count']:,}")
print(f"Total qty:  {row['order_items_custom.quantity_sum']:,}")
print(f"Avg qty:    {row['order_items_custom.quantity_avg']:.2f}")

# Query using the custom weighted_avg aggregation
# No need to pass weight= explicitly — uses the default from the model definition
result = engine.execute(
    query={
        "source_model": "order_items_custom",
        "fields": [
            "quantity:weighted_avg",
        ],
    }
)

row = result.data[0]
print(f"Weighted avg qty: {row['order_items_custom.quantity_weighted_avg']:.2f}")

## Allowed Aggregations Whitelist

Restrict which aggregations make sense for a given measure.

In [ ]:
restricted_model = SlayerModel(
    name="restricted_items",
    sql_table="order_items",
    data_source="jaffle_shop",
    dimensions=[
        Dimension(name="sku", sql="sku", type="string"),
    ],
    measures=[
        Measure(name="quantity", sql="quantity", allowed_aggregations=["sum", "avg", "min", "max"]),
    ],
)
storage.save_model(restricted_model)

# This works: sum is allowed
result = engine.execute(
    query={
        "source_model": "restricted_items",
        "fields": ["quantity:sum"],
    }
)
print(f"quantity:sum = {result.data[0]['restricted_items.quantity_sum']:,}")

# This fails: count_distinct is not in allowed_aggregations
try:
    engine.execute(
        query={
            "source_model": "restricted_items",
            "fields": ["quantity:count_distinct"],
        }
    )
except ValueError as e:
    print(f"\nBlocked as expected: {e}")

## Generated SQL

The colon syntax compiles to standard SQL aggregation functions.

In [ ]:
result = engine.execute(
    query={
        "source_model": "orders",
        "fields": ["*:count", "order_total:sum", "order_total:avg"],
        "dimensions": ["stores.name"],
        "limit": 3,
    }
)

print("Generated SQL:")
print(result.sql)

## Summary

| Concept | Syntax | Example |
|---------|--------|---------|
| Aggregation at query time | `measure:agg` | `revenue:sum`, `price:avg` |
| Row count | `*:count` | Always available, no measure needed |
| Non-null count | `col:count` | `email:count` |
| Distinct count | `col:count_distinct` | `customer_id:count_distinct` |
| First/last by time | `col:first`, `col:last` | `balance:last(updated_at)` |
| Weighted average | `col:weighted_avg(weight=w)` | `price:weighted_avg(weight=qty)` |
| Percentile | `col:percentile(p=0.95)` | `latency:percentile(p=0.99)` |
| Median | `col:median` | Shorthand for percentile(p=0.5) |
| Custom formula | Define in model `aggregations` | `score:trimmed_mean(lo=10, hi=90)` |
| Whitelist | `allowed_aggregations` on measure | Restricts valid aggregations |

See the [aggregations guide](aggregations.md) for the full explanation.